In [0]:
%run ../utils/etl_utils

In [0]:
%python

VOLUME = "/Volumes/dev_dw/bronze/landing_files"

sources = {
    "crm_cust_info":     "source_crm/cust_info.csv",
    "crm_prd_info":      "source_crm/prd_info.csv",
    "crm_sales_details": "source_crm/sales_details.csv",
    "erp_cust_az12":     "source_erp/CUST_AZ12.csv",
    "erp_loc_a101":      "source_erp/LOC_A101.csv",
    "erp_px_cat_g1v2":   "source_erp/PX_CAT_G1V2.csv",
}

def load_csv(table, file):
    df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(f"{VOLUME}/{file}"))
    df.write.format("delta").mode("overwrite").saveAsTable(f"dev_dw.bronze.{table}")
    return df.count()

run_id = new_run_id()
print(f"=== Bronze load | run_id={run_id} ===")

failures = [t for t, f in sources.items()
            if not run_step(run_id, "bronze", t, lambda t=t, f=f: load_csv(t, f))]

if failures:
    raise Exception(f"Bronze load failed: {failures}")

=== Bronze load | run_id=261ccbb2-0df7-42f0-8e61-87129e741afa ===
>> [bronze] crm_cust_info
   OK | 18494 rows | 8.4s
>> [bronze] crm_prd_info
   OK | 397 rows | 4.2s
>> [bronze] crm_sales_details
   OK | 60398 rows | 4.4s
>> [bronze] erp_cust_az12
   OK | 18484 rows | 4.0s
>> [bronze] erp_loc_a101
   OK | 18484 rows | 4.0s
>> [bronze] erp_px_cat_g1v2
   OK | 37 rows | 3.8s
